# HMI Image Stacking Example

Demonstrates solar rotation-corrected image stacking for HMI magnetograms using the egghouse stacking module.

This example covers:
1. Snodgrass differential rotation model
2. StreamingStackAccumulator for memory-efficient processing
3. stack_with_rotation_correction for rotation-corrected stacking
4. Real HMI data download using sunpy Fido (optional)

Requires: `pip install "egghouse[sdo]"`

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
from scipy.ndimage import shift as scipy_shift

# Import stacking module
from egghouse.sdo import (
    snodgrass_rotation_rate,
    solar_rotation_shift,
    StreamingStackAccumulator,
    stack_with_rotation_correction,
    SNODGRASS_A,
    SNODGRASS_B,
    SNODGRASS_C,
    SOLAR_ROTATION_PERIOD,
    HMI_CADENCE_720S,
)

# Check for sunpy availability
try:
    import astropy.units as u
    from sunpy.map import Map
    from sunpy.net import Fido, attrs as a
    HAS_SUNPY = True
except ImportError:
    HAS_SUNPY = False

print(f"sunpy installed: {HAS_SUNPY}")

## Helper Functions

In [ ]:
def create_synthetic_hmi_sequence(
    n_frames: int = 21,
    size: int = 256,
    cadence_seconds: float = 720.0,
    latitude_deg: float = 0.0,
) -> dict:
    """
    Create synthetic HMI-like magnetogram sequence with solar rotation.

    Parameters
    ----------
    n_frames : int
        Number of frames in sequence.
    size : int
        Image size (square).
    cadence_seconds : float
        Time between frames in seconds.
    latitude_deg : float
        Heliographic latitude of the feature.

    Returns
    -------
    dict
        Dictionary containing:
        - images: list of 2D numpy arrays
        - rsun_pixels: solar radius in pixels
        - cadence_hours: cadence in hours
        - latitude_deg: latitude
    """
    # Solar radius in pixels (typical HMI scale)
    rsun_pixels = size * 0.4

    # Calculate rotation rate
    rotation_rate = snodgrass_rotation_rate(latitude_deg)  # deg/day

    # Create coordinate grids
    y, x = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
    center = size // 2

    # Solar disk mask
    r = np.sqrt((x - center)**2 + (y - center)**2)
    disk_mask = r < rsun_pixels

    # Create a bipolar active region pattern
    def create_bipole(x, y, cx, cy, separation=20, strength=500, width=15):
        """Create bipolar magnetic region."""
        # Positive polarity
        pos = strength * np.exp(-((x - cx - separation/2)**2 + (y - cy)**2) / (2 * width**2))
        # Negative polarity
        neg = -strength * np.exp(-((x - cx + separation/2)**2 + (y - cy)**2) / (2 * width**2))
        return pos + neg

    # Initial feature position
    feature_x0 = center - 30  # Start west of center
    feature_y = center + int(latitude_deg * rsun_pixels / 90)  # Approximate

    images = []
    cadence_hours = cadence_seconds / 3600.0

    for i in range(n_frames):
        # Calculate x shift due to rotation
        time_hours = i * cadence_hours
        x_shift = solar_rotation_shift(
            rsun_pixels=rsun_pixels,
            time_offset_hours=time_hours,
            latitude_deg=latitude_deg,
        )

        # Current feature position
        feature_x = feature_x0 + x_shift

        # Create magnetogram
        image = np.zeros((size, size), dtype=np.float32)

        # Add bipolar region
        bipole = create_bipole(x, y, feature_x, feature_y)
        image += bipole

        # Add quiet sun noise
        image += np.random.randn(size, size).astype(np.float32) * 5

        # Apply disk mask
        image = np.where(disk_mask, image, 0)

        images.append(image)

    return {
        "images": images,
        "rsun_pixels": rsun_pixels,
        "cadence_hours": cadence_hours,
        "latitude_deg": latitude_deg,
        "n_frames": n_frames,
    }

## 1. Snodgrass Differential Rotation Model

The Snodgrass (1983) model describes how solar rotation rate varies with latitude:

$$\omega(B) = A + B \cdot \sin^2(B) + C \cdot \sin^4(B)$$

where B is heliographic latitude.

In [ ]:
print(f"Constants:")
print(f"    A = {SNODGRASS_A:.2f} deg/day (equatorial rate)")
print(f"    B = {SNODGRASS_B:.2f} deg/day")
print(f"    C = {SNODGRASS_C:.2f} deg/day")
print(f"\nCarrington rotation period: {SOLAR_ROTATION_PERIOD:.2f} days (~26° latitude)")

In [ ]:
# Calculate rotation rates at different latitudes
print("Rotation rates at different latitudes:")
print("-" * 40)
print(f"{'Latitude':>10} | {'Rate (deg/day)':>15} | {'Period (days)':>13}")
print("-" * 40)

for lat in [0, 15, 30, 45, 60, 75]:
    rate = snodgrass_rotation_rate(lat)
    period = 360.0 / rate
    print(f"{lat:>10}° | {rate:>15.2f} | {period:>13.2f}")

In [ ]:
# Calculate pixel shift example
print("Pixel shift due to rotation (rsun=1600 pixels, 1 hour):")
print("-" * 40)
rsun = 1600  # Typical HMI solar radius

for lat in [0, 30, 60]:
    shift = solar_rotation_shift(rsun, time_offset_hours=1.0, latitude_deg=lat)
    print(f"   Latitude {lat:>2}°: {shift:>6.2f} pixels/hour")

## 2. StreamingStackAccumulator Demo

StreamingStackAccumulator uses Welford's online algorithm for numerically stable incremental computation of mean and variance. Memory usage is O(image_size), independent of number of images.

In [ ]:
# Create synthetic data
print("Creating synthetic sequence (21 frames, 128x128)...")
data = create_synthetic_hmi_sequence(n_frames=21, size=128)

# Initialize accumulator
shape = data["images"][0].shape
accumulator = StreamingStackAccumulator(shape)

print(f"\nAccumulating {len(data['images'])} images...")

for i, image in enumerate(data["images"]):
    accumulator.add(image)

    if (i + 1) % 7 == 0:
        print(f"   After {i+1} images: mean range = "
              f"[{accumulator.get_mean().min():.1f}, {accumulator.get_mean().max():.1f}]")

In [ ]:
# Get final statistics
mean = accumulator.get_mean()
std = accumulator.get_std()

print(f"Final statistics:")
print(f"   Images accumulated: {accumulator.count}")
print(f"   Mean range: [{mean.min():.2f}, {mean.max():.2f}]")
print(f"   Std range: [{std.min():.2f}, {std.max():.2f}]")

# Compare with numpy
numpy_mean = np.mean(data["images"], axis=0)
numpy_std = np.std(data["images"], axis=0, ddof=1)

print(f"\nComparison with numpy (should be nearly identical):")
print(f"   Max mean difference: {np.abs(mean - numpy_mean).max():.2e}")
print(f"   Max std difference: {np.abs(std - numpy_std).max():.2e}")

## 3. Rotation-Corrected Stacking

`stack_with_rotation_correction()` aligns images by compensating for solar rotation before combining them. This preserves spatial features that would otherwise blur due to rotation.

In [ ]:
# Create synthetic sequence at different latitudes
print("Creating sequences at different latitudes...")

for lat in [0, 30]:
    print(f"\n--- Latitude {lat}° ---")

    data = create_synthetic_hmi_sequence(
        n_frames=21,
        size=128,
        cadence_seconds=720.0,
        latitude_deg=lat,
    )

    images = data["images"]
    print(f"   Frames: {len(images)}, Size: {images[0].shape}")
    print(f"   Cadence: {data['cadence_hours']*60:.0f} minutes")

    # Stack without rotation correction (simple mean)
    simple_mean = np.mean(images, axis=0)

    # Stack with rotation correction
    stacked = stack_with_rotation_correction(
        images=images,
        rsun_pixels=data["rsun_pixels"],
        cadence_hours=data["cadence_hours"],
        crop_center=(64, 64),
        crop_size=64,
        latitude_deg=lat,
        method='mean',
    )

    print(f"   Simple mean shape: {simple_mean.shape}")
    print(f"   Rotation-corrected shape: {stacked.shape}")

    # Compare feature sharpness (proxy: max gradient)
    simple_grad = np.sqrt(np.gradient(simple_mean[32:96, 32:96])[0]**2 +
                          np.gradient(simple_mean[32:96, 32:96])[1]**2).max()
    corrected_grad = np.sqrt(np.gradient(stacked)[0]**2 +
                             np.gradient(stacked)[1]**2).max()

    print(f"   Feature sharpness (max gradient):")
    print(f"      Without correction: {simple_grad:.2f}")
    print(f"      With correction: {corrected_grad:.2f}")

## 4. Combining Methods Comparison

Available methods:
- `'list'` - Return all aligned images (no combining)
- `'mean'` - Simple arithmetic mean
- `'median'` - Robust median (outlier resistant)
- `'sigma_clipped'` - Iterative sigma clipping

In [ ]:
# Create data with outliers
print("Creating sequence with simulated cosmic rays...")
data = create_synthetic_hmi_sequence(n_frames=21, size=64)
images = data["images"]

# Add artificial outliers (cosmic ray hits)
np.random.seed(42)
for i in range(5):  # 5 frames with cosmic rays
    frame_idx = np.random.randint(0, len(images))
    y, x = np.random.randint(10, 54, size=2)
    images[frame_idx][y:y+3, x:x+3] = 2000  # Bright spike

# Stack with different methods
results = {}
for method in ['mean', 'median', 'sigma_clipped']:
    stacked = stack_with_rotation_correction(
        images=images,
        rsun_pixels=data["rsun_pixels"],
        cadence_hours=data["cadence_hours"],
        crop_center=(32, 32),
        crop_size=32,
        method=method,
        sigma_lower=3.0,
        sigma_upper=3.0,
    )
    results[method] = stacked

# Compare
print("\nResults (center 32x32 region):")
print("-" * 50)
print(f"{'Method':<15} | {'Max value':<12} | {'Std dev':<12}")
print("-" * 50)

for method, stacked in results.items():
    print(f"{method:<15} | {stacked.max():<12.2f} | {stacked.std():<12.2f}")

print("\nNote: Cosmic rays cause high max values in 'mean'.")
print("'median' and 'sigma_clipped' are more robust to outliers.")

## Typical Usage Patterns

### High-level API: Stacking class (requires sunpy)

```python
from egghouse.sdo import Stacking

# Create stacker
stacker = Stacking(
    nb_stack=21,           # Number of images to stack
    crop_size=512,         # Output size
    method='mean',         # 'mean', 'median', or 'sigma_clipped'
    latitude_deg=0.0,      # For differential rotation
)

# Run on FITS files
file_paths = ['hmi_001.fits', 'hmi_002.fits', ...]
result = stacker.run(file_paths)
```

### Low-level API: stack_with_rotation_correction (numpy arrays)

```python
from egghouse.sdo import stack_with_rotation_correction

# Prepare images as numpy arrays
images = [...]  # List of 2D arrays

# Stack with rotation correction
stacked = stack_with_rotation_correction(
    images=images,
    rsun_pixels=1600,      # Solar radius in pixels
    cadence_hours=0.2,     # 12 minutes
    crop_center=(2048, 2048),
    crop_size=512,
    latitude_deg=15.0,
    method='sigma_clipped',
)
```

### Memory-efficient streaming accumulation

```python
from egghouse.sdo import StreamingStackAccumulator

accumulator = StreamingStackAccumulator(shape=(4096, 4096))

for fits_file in large_file_list:
    image = load_fits(fits_file)
    # Apply rotation correction here if needed
    accumulator.add(image)

mean = accumulator.get_mean()
std = accumulator.get_std()
```

### Snodgrass rotation calculation

```python
from egghouse.sdo import snodgrass_rotation_rate, solar_rotation_shift

# Get rotation rate at latitude
rate = snodgrass_rotation_rate(latitude_deg=30)  # deg/day

# Calculate pixel shift
shift = solar_rotation_shift(
    rsun_pixels=1600,
    time_offset_hours=2.0,  # 2 hours from reference
    latitude_deg=30,
)
```